# Laplace Transform Table

The following table contains the basic Laplace transforms and the most frequently used properties of the unilateral Laplace transform,

$$
F(s)=\mathcal{L}\{f(t)\}=\int_0^\infty f(t)e^{-st}\,dt.
$$

We use **SymPy** to generate the standard transforms and to construct the symbolic identities. The notation follows the original table: $u_c(t)$ denotes a unit step at $t=c$, and $\delta(t-c)$ denotes a Dirac impulse.

In [1]:
from IPython.display import Markdown, display
from sympy import (
    Integer, symbols, Function, Integral, Derivative, Subs, Sum,
    exp, sin, cos, sinh, cosh, gamma, factorial,
    Heaviside, DiracDelta, laplace_transform, latex, simplify,
)

# Symbols and generic functions
t, s = symbols('t s', positive=True)
a, b = symbols('a b', real=True)
c = symbols('c', positive=True)
n = symbols('n', integer=True, nonnegative=True)
k = symbols('k', integer=True, nonnegative=True)
tau = symbols('tau', positive=True)
f, g, F, G = (Function(name) for name in ('f', 'g', 'F', 'G'))

# SymPy's standard one-sided Laplace transform (without convergence data)
def LT(expr):
    return laplace_transform(expr, t, s, noconds=True)

# Present symbolic functions in the compact notation used in the source table.
def fmt(expr):
    return (latex(expr)
        .replace(r'f{\left(t \right)}', r'f(t)')
        .replace(r'g{\left(\tau \right)}', r'g(\tau)')
        .replace(r'F{\left(s \right)}', r'F(s)')
        .replace(r'G{\left(s \right)}', r'G(s)')
        .replace(r'- a + s', r's - a')
        .replace(r'- c + s', r's - c')
        .replace(r'- c + t', r't - c')
        .replace(r'- a^{2} + s^{2}', r's^{2} - a^{2}'))

# A compact format for the n-dependent fractions.
def frac(a, b):
    return r'\frac{' + fmt(a) + '}{' + fmt(b) + '}'

# Rows marked 'computed' come directly from laplace_transform().
# The remaining rows are general transform identities constructed with SymPy.
rows = []
def add(no, time_expr, s_expr, *, time_latex=None, s_latex=None, mode='computed'):
    rows.append(dict(number=no, time_expr=time_expr, s_expr=s_expr,
                     t_tex=time_latex or fmt(time_expr),
                     s_tex=s_latex or fmt(s_expr), mode=mode))

add(1, Integer(1), LT(Integer(1)))
add(2, exp(a*t), LT(exp(a*t)))
add(3, t**n, LT(t**n), s_latex=frac(factorial(n), s**(n+1)))
add(4, t**symbols('p', real=True), LT(t**symbols('p', real=True)),
    time_latex=r't^p\quad(p>-1)',
    s_latex=frac(gamma(symbols('p', real=True)+1), s**(symbols('p', real=True)+1)))
add(5, sin(a*t), LT(sin(a*t)))
add(6, cos(a*t), LT(cos(a*t)))
add(7, sinh(a*t), LT(sinh(a*t)))
add(8, cosh(a*t), LT(cosh(a*t)))
add(9, exp(a*t)*sin(b*t), LT(exp(a*t)*sin(b*t)))
add(10, exp(a*t)*cos(b*t), LT(exp(a*t)*cos(b*t)))
add(11, t**n*exp(a*t), LT(t**n*exp(a*t)),
    s_latex=frac(factorial(n), (s-a)**(n+1)))
add(12, Heaviside(t-c), LT(Heaviside(t-c)), time_latex=r'u_c(t)')
add(13, Heaviside(t-c)*f(t-c), exp(-c*s)*F(s),
    time_latex=r'u_c(t)\,f(t-c)', s_latex=r'e^{-cs}F(s)', mode='property')
add(14, exp(c*t)*f(t), F(s-c), mode='property',
    time_latex=r'e^{ct}f(t)', s_latex=r'F(s-c)')
add(15, f(c*t), F(s/c)/c, mode='property',
    time_latex=r'f(ct)',
    s_latex=r'\frac{1}{c}F\!\left(\frac{s}{c}\right)\quad(c>0)')
add(16, Integral(f(t-tau)*g(tau), (tau, 0, t)), F(s)*G(s), mode='property',
    time_latex=r'\int_0^t f(t-\tau)\,g(\tau)\,d\tau')
add(17, DiracDelta(t-c), LT(DiracDelta(t-c)), time_latex=r'\delta(t-c)')

# General derivative rule: n must be >= 1 in this row; k indexes derivatives.
d_n = Derivative(f(t), (t, n))
initial_sum = Sum(s**(n-1-k) * Subs(Derivative(f(t), (t, k)), t, 0),
                  (k, 0, n-1))
add(18, d_n, s**n*F(s) - initial_sum, mode='property',
    time_latex=r'f^{(n)}(t)',
    s_latex=r's^nF(s)-\sum_{k=0}^{n-1}s^{n-1-k}f^{(k)}(0)')
add(19, (-t)**n*f(t), Derivative(F(s), (s, n)), mode='property',
    time_latex=r'(-t)^n f(t)', s_latex=r'F^{(n)}(s)')

# Render a Markdown table with MathJax math expressions.
header = [r'| No. | $f(t)=\mathcal{L}^{-1}\{F(s)\}$ | $F(s)=\mathcal{L}\{f(t)\}$ |',
          '|:---:|:---:|:---:|']
lines = [f"| {row['number']} | ${row['t_tex']}$ | ${row['s_tex']}$ |" for row in rows]
laplace_table_markdown = '\n'.join(header + lines)
display(Markdown(laplace_table_markdown))

| No. | $f(t)=\mathcal{L}^{-1}\{F(s)\}$ | $F(s)=\mathcal{L}\{f(t)\}$ |
|:---:|:---:|:---:|
| 1 | $1$ | $\frac{1}{s}$ |
| 2 | $e^{a t}$ | $\frac{1}{s - a}$ |
| 3 | $t^{n}$ | $\frac{n!}{s^{n + 1}}$ |
| 4 | $t^p\quad(p>-1)$ | $\frac{\Gamma\left(p + 1\right)}{s^{p + 1}}$ |
| 5 | $\sin{\left(a t \right)}$ | $\frac{a}{a^{2} + s^{2}}$ |
| 6 | $\cos{\left(a t \right)}$ | $\frac{s}{a^{2} + s^{2}}$ |
| 7 | $\sinh{\left(a t \right)}$ | $\frac{a}{s^{2} - a^{2}}$ |
| 8 | $\cosh{\left(a t \right)}$ | $\frac{s}{s^{2} - a^{2}}$ |
| 9 | $e^{a t} \sin{\left(b t \right)}$ | $\frac{b}{b^{2} + \left(s - a\right)^{2}}$ |
| 10 | $e^{a t} \cos{\left(b t \right)}$ | $\frac{s - a}{b^{2} + \left(s - a\right)^{2}}$ |
| 11 | $t^{n} e^{a t}$ | $\frac{n!}{\left(s - a\right)^{n + 1}}$ |
| 12 | $u_c(t)$ | $\frac{e^{- c s}}{s}$ |
| 13 | $u_c(t)\,f(t-c)$ | $e^{-cs}F(s)$ |
| 14 | $e^{ct}f(t)$ | $F(s-c)$ |
| 15 | $f(ct)$ | $\frac{1}{c}F\!\left(\frac{s}{c}\right)\quad(c>0)$ |
| 16 | $\int_0^t f(t-\tau)\,g(\tau)\,d\tau$ | $F(s) G(s)$ |
| 17 | $\delta(t-c)$ | $e^{- c s}$ |
| 18 | $f^{(n)}(t)$ | $s^nF(s)-\sum_{k=0}^{n-1}s^{n-1-k}f^{(k)}(0)$ |
| 19 | $(-t)^n f(t)$ | $F^{(n)}(s)$ |

### Notes

- $n=0,1,2,\ldots$ in the power and multiplication rules; $n\ge 1$ in the differentiation rule.
- $p>-1$ and $c>0$ where indicated. The region of convergence of $F(s)$ must also be respected; the table omits those details for brevity.
- Rows **1–12 and 17** are calculated directly with SymPy's `laplace_transform`.
- Rows **13–16 and 18–19** are general properties for an arbitrary $f(t)$ (and $g(t)$). SymPy represents these using symbolic functions, products, integrals, derivatives, and sums, rather than automatically evaluating every generic transform.

In [2]:
# Check that the entries computed directly from SymPy agree with the table.
for row in rows:
    if row['mode'] == 'computed':
        assert simplify(LT(row['time_expr']) - row['s_expr']) == 0, row['number']
print('Verified', sum(row['mode'] == 'computed' for row in rows),
      'direct Laplace transforms with SymPy.')

Verified 13 direct Laplace transforms with SymPy.


### Checking some transform properties

We can test the general properties by choosing a simple example,

$$f(t)=e^{-2t},\qquad F(s)=\frac{1}{s+2}.$$

For example, the time-delay property states that

$$\mathcal{L}\{u_c(t)f(t-c)\}=e^{-cs}F(s).$$

The following calculations verify the time-delay, frequency-shift, time-scaling, and convolution rules for this example.

In [3]:
from sympy import integrate

F_example = 1/(s+2)
G_example = 1/(s+3)
f_example = exp(-2*t)
g_example = exp(-3*t)

checks = {
    'time shift': (LT(Heaviside(t-c)*exp(-2*(t-c))), exp(-c*s)*F_example),
    'frequency shift': (LT(exp(c*t)*f_example), F_example.subs(s, s-c)),
    'time scaling': (LT(exp(-2*c*t)), F_example.subs(s, s/c)/c),
    'convolution': (
        LT(integrate(exp(-2*(t-tau))*exp(-3*tau), (tau, 0, t))),
        F_example*G_example,
    ),
}
for label, (actual, expected) in checks.items():
    assert simplify(actual - expected) == 0, label
    print(f'{label}: PASS')

# Check the differentiation and multiplication rules at order 2.
d2 = LT(f_example.diff(t, 2))
assert simplify(d2 - (s**2*F_example - s*Integer(1) + Integer(2))) == 0
assert simplify(LT(t**2*f_example) - F_example.diff(s, 2)) == 0
print('second derivative: PASS')
print('multiplication by t^2: PASS')

time shift: PASS
frequency shift: PASS
time scaling: PASS
convolution: PASS
second derivative: PASS
multiplication by t^2: PASS


## References

https://www.math.purdue.edu/~liu2053/Formula_sheets_for_final_exam.pdf